In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 2 と Qiskit SDK

<details>
<summary><b>Package versions</b></summary>

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降を使用することを推奨します。

```
qiskit[all]~=2.3.0
```
</details>

Qiskit SDK は、量子プログラムの OpenQASM 表現と [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) クラスの間で変換するためのツールを提供しています。

<span id="qasm2-import"></span>
## OpenQASM 2 プログラムを Qiskit にインポートする
OpenQASM 2 プログラムを Qiskit にインポートする関数が 2 つあります。
ファイル名を受け取る [`qasm2.load()`](../api/qiskit/qasm2#load) と、OpenQASM 2 プログラムを文字列として受け取る [`qasm2.loads()`](../api/qiskit/qasm2#loads) です。

In [1]:
import qiskit.qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";
    qreg q[2];
    creg c[2];

    h q[0];
    cx q[0], q[1];

    measure q -> c;
"""
circuit = qiskit.qasm2.loads(program)
circuit.draw()

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

詳細については、[OpenQASM 2 Qiskit API](https://docs.quantum.ibm.com/api/qiskit/qasm2) を参照してください。

### シンプルなプログラムのインポート
ほとんどの OpenQASM 2 プログラムでは、`qasm2.load` と `qasm2.loads` を引数一つで使用するだけで問題ありません。

#### 例：OpenQASM 2 プログラムを文字列としてインポートする
`qasm2.loads()` を使用して、OpenQASM 2 プログラムを文字列として QuantumCircuit にインポートします：

In [2]:
from qiskit import qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";

    qreg q[4];
    creg c[4];

    h q[0];
    cx q[0], q[1];

    // 'rxx' is not actually in `qelib1.inc`,
    // but Qiskit used to behave as if it were.
    rxx(0.75) q[2], q[3];

    measure q -> c;
"""
circuit = qasm2.loads(
    program,
    custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
)

#### Example: use a particular gate class when importing an OpenQASM 2 program

Qiskit cannot, in general, verify if the definition in an OpenQASM 2 `gate` statement corresponds exactly to a Qiskit standard-library gate.
Instead, Qiskit chooses a custom gate using the precise definition supplied.
This can be less efficient that using one of the built-in standard gates, or a user-defined custom gate.
You can manually define `gate` statements with particular classes.

In [3]:
from qiskit import qasm2
from qiskit.circuit import Gate
from qiskit.circuit.library import RZXGate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    # Link the OpenQASM 2 name 'my' with our custom gate.
    qasm2.CustomInstruction("my", 2, 1, MyGate),
    # Link the OpenQASM 2 name 'rzx' with Qiskit's
    # built-in RZXGate.
    qasm2.CustomInstruction("rzx", 1, 2, RZXGate),
]

program = """
    OPENQASM 2.0;

    gate my(theta, phi) q {
        U(theta / 2, phi, -theta / 2) q;
    }
    gate rzx(theta) a, b {
        // It doesn't matter what definition is
        // supplied, if the parameters match;
        // Qiskit will still use `RZXGate`.
    }

    qreg q[2];
    my(0.25, 0.125) q[0];
    rzx(pi) q[0], q[1];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

#### 例：OpenQASM 2 プログラムをファイルからインポートする
`load()` を使用して、OpenQASM 2 プログラムをファイルから QuantumCircuit にインポートします：

In [4]:
from qiskit import qasm2
from qiskit.circuit import Gate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    qasm2.CustomInstruction("my", 2, 1, MyGate, builtin=True),
]

program = """
    OPENQASM 2.0;
    qreg q[1];

    my(0.25, 0.125) q[0];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

<span id="custom-instructions"></span>
### OpenQASM 2 ゲートを Qiskit ゲートにリンクする
デフォルトでは、Qiskit の OpenQASM 2 インポーターはインクルードファイル `"qelib1.inc"` を*事実上の*標準ライブラリとして扱います。
インポーターは、このファイルが [OpenQASM 2 を定義したオリジナルの論文](https://arxiv.org/abs/1707.03429)で説明されているゲートを正確に含むものとして扱います。
Qiskit は `"qelib1.inc"` 内のゲートを表現するために [回路ライブラリ](../api/qiskit/circuit_library) の組み込みゲートを使用します。
プログラム内で手動の OpenQASM 2 `gate` 文によって定義されたゲートは、デフォルトでカスタムの [Qiskit `Gate` サブクラス](../api/qiskit/qiskit.circuit.Gate) として構築されます。

インポーターが検出する特定の `gate` 文に対して使用する [`Gate`](../api/qiskit/qiskit.circuit.Gate) クラスを指定することができます。
このメカニズムを使用して、追加のゲート名を「組み込み」として扱うこともできます。つまり、明示的な定義を必要としないということです。
`"qelib1.inc"` 以外の `gate` 文に対して使用するゲートクラスを指定すると、生成される回路は通常より効率的に扱えるようになります。

> **Warning:** Qiskit SDK v1.0 以降、Qiskit の OpenQASM 2 *エクスポーター*（[Qiskit 回路を OpenQASM 2 にエクスポートする](#qasm2-export) を参照）は、`"qelib1.inc"` に実際よりも多くのゲートがあるかのように動作し続けています。
> つまり、インポーターのデフォルト設定では、自身のエクスポーターによってエクスポートされたプログラムをインポートできない場合があります。
> この問題を解決するには、[レガシーエクスポーターとの連携に関する具体的な例](#qasm2-import-legacy) を参照してください。
> 
> この不整合は Qiskit のレガシーな動作であり、[Qiskit の今後のリリースで解決される予定です](https://github.com/Qiskit/qiskit/issues/10737)。

カスタム命令に関する情報を OpenQASM 2 インポーターに渡すには、[`qasm2.CustomInstruction` クラス](../api/qiskit/qasm2#qiskit.qasm2.CustomInstruction) を使用します。
このクラスには、順番に 4 つの必須情報があります：

* OpenQASM 2 プログラムで使用されるゲートの**名前**
* ゲートが受け取る**角度パラメーター数**
* ゲートが作用する**量子ビット数**
* ゲートの Python **コンストラクター**クラスまたは関数（量子ビットではなく、ゲートパラメーターを個別の引数として受け取る）

インポーターが特定のカスタム命令に一致する `gate` 定義を検出すると、そのカスタム情報を使用してゲートオブジェクトを再構築します。
カスタム命令の `name` に一致する `gate` 文が検出されたが、パラメーター数と量子ビット数の両方が一致しない場合、インポーターは [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror) を発生させ、提供された情報とプログラムの不一致を示します。

さらに、5 番目の引数 `builtin` をオプションで `True` に設定すると、OpenQASM 2 プログラム内で明示的に定義されていなくてもゲートが自動的に利用可能になります。
インポーターが組み込みのカスタム命令に対する明示的な `gate` 定義を検出した場合、それを無視します。
上記と同様に、同じ名前の明示的な定義が提供されたカスタム命令と互換性がない場合、[`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror) が発生します。
これは、古い OpenQASM 2 エクスポーターとの互換性や、ハードウェアの「基底ゲート」を組み込み命令として扱う特定の量子プラットフォームとの互換性に役立ちます。

Qiskit は、[Qiskit の OpenQASM 2 エクスポート機能](#qasm2-export) のレガシーバージョンによって生成された OpenQASM 2 プログラムを扱うためのデータ属性を提供しています。
これは [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility) であり、[`qasm2.load()`](../api/qiskit/qasm2#load) と [`qasm2.loads()`](../api/qiskit/qasm2#loads) の `custom_instructions` 引数として渡すことができます。

<span id="qasm2-import-legacy"></span>
#### 例：Qiskit のレガシーエクスポーターで作成されたプログラムをインポートする
この OpenQASM 2 プログラムは、`"qelib1.inc"` のオリジナルバージョンには含まれていないゲートを宣言なしで使用していますが、これらは Qiskit のライブラリの標準ゲートです。
[`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility) を使用することで、Qiskit の OpenQASM 2 エクスポーターが以前使用していたゲートセットを使用するようインポーターに簡単に伝えることができます。

In [5]:
import math
from qiskit import qasm2

program = """
    include "qelib1.inc";
    qreg q[2];
    rx(arctan(pi, 3 + add_one(0.2))) q[0];
    cx q[0], q[1];
"""


def add_one(x):
    return x + 1


customs = [
    # Our `add_one` takes only one parameter.
    qasm2.CustomClassical("add_one", 1, add_one),
    # `arctan` takes two parameters, and `math.atan2` implements it.
    qasm2.CustomClassical("arctan", 2, math.atan2),
]
circuit = qasm2.loads(program, custom_classical=customs)

#### 例：OpenQASM 2 プログラムをインポートする際に特定のゲートクラスを使用する
Qiskit は一般に、OpenQASM 2 の `gate` 文の定義が Qiskit の標準ライブラリゲートと完全に対応するかどうかを検証することができません。
代わりに、Qiskit は提供された正確な定義を使用してカスタムゲートを選択します。
これは、組み込みの標準ゲートやユーザー定義のカスタムゲートを使用するよりも効率が低い場合があります。
特定のクラスで `gate` 文を手動で定義することができます。

In [6]:
from qiskit import QuantumCircuit, qasm2

# Define any circuit.
circuit = QuantumCircuit(2, 2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure([0, 1], [0, 1])

# Export to a string.
program = qasm2.dumps(circuit)

# Export to a file.
qasm2.dump(circuit, "my_file.qasm")

#### 例：OpenQASM 2 プログラムに新しい組み込みゲートを定義する
引数 `builtin=True` を設定すると、カスタムゲートに関連する定義がなくても構いません。